# T&D — Priorização de Melhoria Física por Produto
**Insider Store · Supply Chain / SOP**

Diagnóstico de Trocas e Devoluções cruzando dor do cliente, tags qualitativas e tração comercial.  
Período: últimos 12 meses · Data de referência: 2026-06-11.

**Saídas produzidas:**
- Farol executivo por produto (`sinal_priorizacao` + `priority_score`)
- Benchmark por categoria
- Top problemas por tag
- Split Físico vs Logístico
- Export para Excel + CSV

> **Query:** `pipeline_reversas_priorizacao_produtos.sql`  
> **Funções:** `td_analysis_functions.py`

In [1]:
# === Setup ===================================================================
import sys
from pathlib import Path
from datetime import date

import pandas as pd
import numpy as np
from google.cloud import bigquery
from IPython.display import display

ANALYSIS_DIR = Path('/Users/insider/LA_Coding_Projects/analyses/relatorio_td')
OUTPUT_DIR   = Path('/Users/insider/LA_Coding_Projects/outputs/relatorio_td')
SQL_FILE     = ANALYSIS_DIR / 'pipeline_reversas_priorizacao_produtos.sql'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Garante que o módulo é recarregado após qualquer edição
for _key in list(sys.modules.keys()):
    if 'td_analysis_functions' in _key:
        del sys.modules[_key]

if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from td_analysis_functions import (
    prepare_executive_df,
    classify_priority,
    priority_summary,
    top_offenders,
    category_benchmark,
    tag_summary_from_executive,
    validate_no_tag_duplication,
    problema_tipo_split,
    build_llm_context_rows,
    build_tweet_heuristic,
    build_mensagem_consolidada,
    build_scorecard_product_view,
    build_tweets_tab,
    export_priority_workbook,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

DATE_TAG = date.today().strftime('%Y%m%d')
print(f'Setup OK | output: {OUTPUT_DIR} | tag: {DATE_TAG}')


Setup OK | output: /Users/insider/LA_Coding_Projects/outputs/relatorio_td | tag: 20260612


In [2]:
xlsx_path = str(OUTPUT_DIR / f'td_priorizacao_{DATE_TAG}.xlsx')
print(f'xlsx_path: {xlsx_path}')


xlsx_path: /Users/insider/LA_Coding_Projects/outputs/relatorio_td/td_priorizacao_20260612.xlsx


In [3]:
PROJECT_ID = 'insider-data-lake'
client = bigquery.Client(project=PROJECT_ID)
print(f'BigQuery client OK — projeto: {PROJECT_ID}')

/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


BigQuery client OK — projeto: insider-data-lake


In [4]:
# Carrega e divide as duas queries do arquivo SQL.
# O arquivo contém: Camada 2 (executiva) primeiro, depois Camada 1 (analítica).
sql_raw = SQL_FILE.read_text(encoding='utf-8')

CAMADA1_HEADER = '-- CAMADA 1: TABELA ANALÍTICA ITEM / REVERSA / TAG'
idx = sql_raw.find(CAMADA1_HEADER)
if idx == -1:
    raise ValueError(f'Header da Camada 1 não encontrado em {SQL_FILE}')

def _extract_query(block: str) -> str:
    start = block.find('WITH ')
    return block[start:].strip() if start != -1 else block.strip()

SQL_EXECUTIVO = _extract_query(sql_raw[:idx])
SQL_ANALITICO = _extract_query(sql_raw[idx:])

print(f'SQL carregado: {SQL_FILE}')
print(f'  Camada 2 (executiva): {len(SQL_EXECUTIVO):,} chars')
print(f'  Camada 1 (analítica): {len(SQL_ANALITICO):,} chars')

SQL carregado: /Users/insider/LA_Coding_Projects/analyses/relatorio_td/pipeline_reversas_priorizacao_produtos.sql
  Camada 2 (executiva): 25,535 chars
  Camada 1 (analítica): 10,990 chars


## Execução das Queries BigQuery

In [5]:
print('Executando Camada 2 — Tabela Executiva...')
df_exec = client.query(SQL_EXECUTIVO).to_dataframe()
print(f'Camada 2: {len(df_exec):,} produto(s) x {df_exec.shape[1]} colunas')
df_exec.head(3)

Executando Camada 2 — Tabela Executiva...


/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Camada 2: 115 produto(s) x 44 colunas


,product_name,category,gender,portfolio_cluster,sinal_priorizacao,priority_score,td_score,commercial_score,qt_pedidos,qt_skus,qt_items_vendidos,receita_liquida,qt_reversas,qt_items_returned,qt_trocas,qt_devolucoes,qt_reversas_fisico,qt_reversas_logistico,valor_troca,valor_devolucao,td_rate,td_rate_categoria,delta_vs_categoria,ratio_vs_categoria,share_receita_portfolio,share_unidades_portfolio,share_td_portfolio,top_1_problema,top_1_pct,top_2_problema,top_2_pct,top_3_problema,top_3_pct,pct_top_3_total,principal_cor_afetada,principal_cor_pct,principal_tamanho_afetado,principal_tamanho_pct,reversas_ultimos_3m,reversas_3m_anteriores,tendencia_reversas,comentarios_amostra,resumo_pre_llm,portfolio_cluster_payload
0,Calça FutureForm Feminino,Calça,female,Hero,Priorizar melhoria,0.9054,0.9200,0.8835,24971,15,26018,8996127.220000000,4287,"4,293.0000",2528,1703,3813,15,"1,069,425.8200","961,715.1800",0.1650,0.1230,0.0420,1.3420,0.025647305,0.0087,0.0251,caimento_ruim,0.1442,modelagem_ruim,0.1225,tamanho_pequeno,0.0485,0.3151,Preto,0.5419,M,0.3508,1194,1227,Estável,A aniversariante disse que parece uma lona de ...,"Top problemas: caimento_ruim, modelagem_ruim, ...","{""cluster"":""Hero"",""product_name"":""Calça Future..."
1,Tube Dress Feminino,Vestido,female,Core,Priorizar melhoria,0.8628,0.8861,0.8278,20346,5,20465,5699452.900000000,3511,"3,514.0000",2132,1344,3231,4,"700,761.5900","587,155.2460",0.1717,0.1634,0.0083,1.0509,0.016248726,0.0068,0.0205,tamanho_pequeno,0.1589,caimento_ruim,0.1196,modelagem_ruim,0.0661,0.3446,Preto,1.0000,M,0.3908,754,1036,Caiu nos últimos 3 meses,muito justo o vestido p meu corpo || Tenho si...,"Top problemas: tamanho_pequeno, caimento_ruim,...","{""cluster"":""Core"",""product_name"":""Tube Dress F..."
2,Skin Cropped Long Sleeve Feminino,Cropped,female,Long tail,Priorizar melhoria,0.8264,0.9113,0.6991,16038,12,18649,2178086.700000000,2904,"2,961.0000",1852,1001,2763,6,"432,332.7950","342,993.4250",0.1588,0.1043,0.0545,1.5228,0.006209567,0.0062,0.0173,tamanho_pequeno,0.1374,modelagem_ruim,0.0379,caimento_ruim,0.0368,0.2121,Preto,0.7183,M,0.3953,1704,292,Aumentou nos últimos 3 meses,O modelo é justo e não gostei || Sou cliente i...,"Top problemas: tamanho_pequeno, modelagem_ruim...","{""cluster"":""Long tail"",""product_name"":""Skin Cr..."


In [6]:
# Camada 1 pode ser demorada — defina LOAD_ANALITICA = False para pular
LOAD_ANALITICA = False

if LOAD_ANALITICA:
    print('Executando Camada 1 — Base Analítica (pode demorar)...')
    df_analitica = client.query(SQL_ANALITICO).to_dataframe()
    print(f'Camada 1: {len(df_analitica):,} linhas (order/sku/tag)')
    display(df_analitica.head(3))
else:
    df_analitica = None
    print('Camada 1 pulada (LOAD_ANALITICA = False)')


Camada 1 pulada (LOAD_ANALITICA = False)


## QA — Verificações de Integridade

Dois checks obrigatórios antes de usar os resultados:
1. **Labels Python vs SQL** — recomputa `sinal_priorizacao` em Python e compara com o valor vindo do BigQuery.
2. **Anti-duplicação de tags** — confirma que o numerador T&D não foi inflado por múltiplas tags por reversa.

> **Metodologia — Tração Comercial (Opção B):** `commercial_score` é o proxy interno de Tração Comercial, calculado como `0.6 × percentil_receita + 0.4 × percentil_unidades`. Não há join externo com scorecard de pilares. O mapeamento de sinal QA (`sinal_kill_keep`) aplica a regra de dois estados — **Alta T&D + Alta Tração → Priorizar melhoria** / **Alta T&D + Baixa Tração → Não Priorizar (Avaliar Descontinuação/Reformulação)** — sobre os scores já computados pelo SQL. Ver seção 9 para o cruzamento completo.

In [7]:
# QA 1: Labels Python vs SQL
df_classified = classify_priority(df_exec)
divergencias = df_classified[~df_classified['sinal_confere_sql']]
n_div = len(divergencias)
status = 'OK — 100% conferem' if n_div == 0 else f'ATENCAO — {n_div} divergencia(s)'
print(f'Labels Python vs SQL: {status}')
if not divergencias.empty:
    display(
        divergencias[
            ['product_name', 'sinal_priorizacao', 'sinal_priorizacao_python',
             'td_score', 'commercial_score']
        ].head(20)
    )

Labels Python vs SQL: OK — 100% conferem


In [8]:
# QA 2: Anti-duplicação de tags no numerador T&D
if df_analitica is not None:
    qa = validate_no_tag_duplication(df_analitica, df_exec)
    print('Validacao anti-duplicacao de tags:')
    for k, v in qa.items():
        ok_flag = '' if k != 'passes' else (' OK' if v else ' FALHOU')
        print(f'  {k}: {v}{ok_flag}')
else:
    print('QA de duplicacao pulado — Camada 1 nao carregada')

QA de duplicacao pulado — Camada 1 nao carregada


## Diagnósticos

### 1. Resumo do Portfólio por Farol

In [9]:
df_summary = priority_summary(df_exec).copy()

for col, fmt in [
    ('td_rate_medio',           '{:.2%}'),
    ('share_receita',           '{:.1%}'),
    ('share_itens_retornados',  '{:.1%}'),
    ('priority_score_medio',    '{:.3f}'),
]:
    if col in df_summary.columns:
        df_summary[col] = df_summary[col].map(fmt.format)

if 'receita_liquida' in df_summary.columns:
    df_summary['receita_liquida'] = df_summary['receita_liquida'].map('R$ {:,.0f}'.format)

df_summary.rename(columns={
    'sinal_priorizacao':       'Farol',
    'produtos':                'Produtos',
    'unidades_vendidas':       'Unid. Vendidas',
    'receita_liquida':         'Receita Liquida',
    'itens_retornados':        'Itens Retornados',
    'td_rate_medio':           'TD Rate Medio',
    'share_receita':           'Share Receita',
    'share_itens_retornados':  'Share T&D',
}, inplace=True)

display(df_summary)

,Farol,Produtos,Unid. Vendidas,Receita Liquida,Itens Retornados,TD Rate Medio,priority_score_medio,Share Receita,Share T&D
0,Alerta em produto relevante,13,1276637,"R$ 174,596,249","80,230.0000",7.75%,0.706,49.8%,46.9%
1,Não priorizar agora,71,1299643,"R$ 91,296,320","39,268.0000",5.87%,0.399,26.0%,22.9%
2,Priorizar melhoria,12,296642,"R$ 65,129,757","37,392.0000",14.94%,0.813,18.6%,21.8%
3,Monitorar,18,115656,"R$ 19,739,288","14,298.0000",14.35%,0.596,5.6%,8.4%
4,Sem evidência suficiente,1,6,"R$ 1,446",0.0000,0.00%,0.009,0.0%,0.0%


### 2. Top Produtos — Priorizar Melhoria

In [10]:
df_top = top_offenders(df_exec, bucket='Priorizar melhoria', n=20)
cols = [
    'product_name', 'category', 'gender', 'portfolio_cluster',
    'td_rate', 'td_rate_categoria', 'delta_vs_categoria',
    'qt_items_returned', 'qt_reversas_fisico', 'qt_reversas_logistico',
    'receita_liquida', 'priority_score',
    'top_1_problema', 'top_1_pct', 'tendencia_reversas',
]
available = [c for c in cols if c in df_top.columns]
print(f'Priorizar melhoria: {len(df_top)} produto(s)')
display(df_top[available].reset_index(drop=True))

Priorizar melhoria: 12 produto(s)


,product_name,category,gender,portfolio_cluster,td_rate,td_rate_categoria,delta_vs_categoria,qt_items_returned,qt_reversas_fisico,qt_reversas_logistico,receita_liquida,priority_score,top_1_problema,top_1_pct,tendencia_reversas
0,Calça FutureForm Feminino,Calça,female,Hero,0.1650,0.1230,0.0420,"4,293.0000",3813,15,"8,996,127.2200",0.9054,caimento_ruim,0.1442,Estável
1,Tube Dress Feminino,Vestido,female,Core,0.1717,0.1634,0.0083,"3,514.0000",3231,4,"5,699,452.9000",0.8628,tamanho_pequeno,0.1589,Caiu nos últimos 3 meses
2,Skin Cropped Long Sleeve Feminino,Cropped,female,Long tail,0.1588,0.1043,0.0545,"2,961.0000",2763,6,"2,178,086.7000",0.8264,tamanho_pequeno,0.1374,Aumentou nos últimos 3 meses
3,Saia Mini Kyoto Feminino,Saia,female,Core,0.2045,0.1190,0.0855,"2,315.0000",2063,3,"2,620,113.0900",0.8243,caimento_ruim,0.1263,Aumentou nos últimos 3 meses
4,Calça Wide Leg Feminino,Calça,female,Core,0.1303,0.1230,0.0074,"2,599.0000",2346,17,"5,486,871.5500",0.8134,caimento_ruim,0.0781,Aumentou nos últimos 3 meses
5,Core T-shirt Masculino,T-shirt,male,Hero,0.0779,0.0550,0.0230,"8,429.0000",7063,41,"13,981,701.9200",0.8127,tamanho_pequeno,0.0617,Estável
6,Camisa FutureForm Feminino,Camisa Social,female,Core,0.1389,0.1421,-0.0032,"3,023.0000",2771,6,"7,947,348.6700",0.8118,tamanho_grande,0.1059,Aumentou nos últimos 3 meses
7,Shorts Kyoto Feminino,Shorts,female,Core,0.1841,0.1841,0.0000,"2,906.0000",2690,9,"3,425,742.1200",0.8077,caimento_ruim,0.1094,Estável
8,Camisa FutureForm Masculino,Camisa Social,male,Core,0.1522,0.1421,0.0101,"2,219.0000",2075,6,"5,173,195.0200",0.8077,tamanho_grande,0.1270,Aumentou nos últimos 3 meses
9,Bermuda Kyoto Feminino,Bermuda,female,Core,0.1699,0.1327,0.0372,"1,628.0000",1469,5,"2,514,127.1300",0.7743,tamanho_grande,0.1002,Estável


### 3. Alerta em Produto Relevante

In [11]:
df_alerta = top_offenders(df_exec, bucket='Alerta em produto relevante', n=15)
cols_alerta = [
    'product_name', 'category', 'portfolio_cluster',
    'td_rate', 'td_rate_categoria', 'delta_vs_categoria',
    'receita_liquida', 'share_receita_portfolio',
    'qt_items_returned', 'commercial_score', 'td_score',
    'top_1_problema',
]
available_alerta = [c for c in cols_alerta if c in df_alerta.columns]
print(f'Alerta em produto relevante: {len(df_alerta)} produto(s)')
display(df_alerta[available_alerta].reset_index(drop=True))

Alerta em produto relevante: 13 produto(s)


,product_name,category,portfolio_cluster,td_rate,td_rate_categoria,delta_vs_categoria,receita_liquida,share_receita_portfolio,qt_items_returned,commercial_score,td_score,top_1_problema
0,The Perfect Top Feminino,Regata,Long tail,0.0749,0.0766,-0.0016,"32,806,350.4900",0.0935,"23,507.0000",0.9913,0.6357,tamanho_pequeno
1,Tech T-shirt Gola U Masculino,T-shirt,Hero,0.0510,0.0550,-0.0040,"59,045,621.7100",0.1683,"28,416.0000",1.0000,0.5583,tamanho_pequeno
2,Maxi Saia NYIN Feminino,Saia,Hero,0.0917,0.1190,-0.0273,"11,598,761.5300",0.0331,"3,376.0000",0.9183,0.5904,caimento_ruim
3,Tech T-shirt Heavy Masculino,T-shirt,Core,0.0703,0.0550,0.0154,"4,957,516.0900",0.0141,"2,443.0000",0.8435,0.6400,tamanho_pequeno
4,Calça FutureForm Masculino,Calça,Hero,0.0927,0.1230,-0.0303,"12,163,749.3700",0.0347,"3,301.0000",0.9200,0.5861,tamanho_pequeno
5,Wingsuit Feminino,Casaco,Hero,0.0534,0.0535,-0.0001,"17,335,360.6000",0.0494,"2,629.0000",0.9443,0.5557,caimento_ruim
6,Camiseta Polo Core Masculino,T-shirt,Hero,0.0558,0.0550,0.0009,"8,745,218.0700",0.0249,"2,550.0000",0.9096,0.5687,tamanho_grande
7,Daily T-shirt Feminino,T-shirt,Long tail,0.0822,0.0550,0.0272,"2,411,705.1600",0.0069,"2,018.0000",0.7426,0.6757,tamanho_pequeno
8,Daily T-shirt Masculino,T-shirt,Hero,0.0504,0.0550,-0.0046,"10,130,011.3800",0.0289,"5,232.0000",0.9513,0.5287,tamanho_pequeno
9,Camiseta Henley Core Masculino,T-shirt,Core,0.0642,0.0550,0.0092,"5,304,641.6800",0.0151,"1,859.0000",0.8435,0.5904,tamanho_grande


### 4. Benchmark por Categoria

In [12]:
df_cat = category_benchmark(df_exec)
display(df_cat)

,category,produtos,unidades_vendidas,receita_liquida,itens_retornados,produtos_priorizar,td_rate_medio_produto,td_rate_mediano_produto,td_rate_categoria_recalculado
0,T-shirt,30,1247758,"147,987,382.5100","68,587.0000",2,0.0702,0.0615,0.0550
1,Calça,5,89456,"29,423,074.5200","10,999.0000",2,0.1211,0.1185,0.1230
2,Saia,5,74176,"21,671,900.2500","8,827.0000",2,0.1271,0.1125,0.1190
3,Camisa Social,3,38343,"13,643,923.4800","5,449.0000",2,0.1315,0.1389,0.1421
4,Cropped,6,76921,"6,408,699.2700","8,020.0000",1,0.1042,0.0959,0.1043
5,Vestido,7,43094,"12,808,647.7100","7,041.0000",1,0.1563,0.1486,0.1634
6,Bermuda,2,24914,"6,132,700.1300","3,307.0000",1,0.1397,0.1397,0.1327
7,Shorts,1,15785,"3,425,742.1200","2,906.0000",1,0.1841,0.1841,0.1841
8,Regata,4,335338,"36,255,553.3000","25,670.0000",0,0.0892,0.0899,0.0765
9,Cueca,6,415660,"23,331,629.5800","9,403.0000",0,0.0243,0.0210,0.0226


### 5. Top Problemas por Tag

In [13]:
df_tags = tag_summary_from_executive(df_exec)
display(df_tags.head(20))

,problema_tag,produtos,produtos_priorizar,pct_medio_no_produto
0,caimento_ruim,87,12,0.0881
1,modelagem_ruim,68,10,0.0773
2,tamanho_pequeno,56,7,0.0943
3,tamanho_grande,41,5,0.0810
4,cor_diferente_site,5,1,0.0489
5,tecido_grosso,4,1,0.0580
6,tecido_qualidade_ruim,16,0,0.0519
7,conforto_negativo,12,0,0.0949
8,defeito_costura,12,0,0.0541
9,defeito_furo_rasgo,6,0,0.0976


### 6. Split por Tipo de Problema — Físico vs Logístico

In [14]:
df_tipo = problema_tipo_split(df_exec).copy()
df_tipo['pct_total'] = df_tipo['pct_total'].map('{:.1%}'.format)
display(df_tipo)

,tipo_problema,qt_reversas,pct_total
0,Físico,"141,995.0000",91.1%
1,Outros / Desistência,"12,921.0000",8.3%
2,Logístico,927.0000,0.6%


### 7. Tendência de Reversas (últimos 3 meses vs 3 meses anteriores)

In [15]:
df_prep = prepare_executive_df(df_exec)

# Mapeamento QA spec: Alta T&D + Alta Tração → Priorizar | Alta T&D + Baixa Tração → Avaliar Descontinuação
df_prep['sinal_kill_keep'] = np.select(
    [
        (df_prep['qt_items_vendidos'] < 30) | (df_prep['qt_items_returned'] < 5),
        (df_prep['td_score'] >= 0.70) & (df_prep['commercial_score'] >= 0.60),
        (df_prep['td_score'] >= 0.70) & (df_prep['commercial_score'] < 0.60),
    ],
    [
        'Sem evidência suficiente',
        'Priorizar melhoria',
        'Não Priorizar (Avaliar Descontinuação/Reformulação)',
    ],
    default='Não priorizar agora',
)

if 'tendencia_reversas' in df_prep.columns:
    df_trend = (
        df_prep
        .groupby('tendencia_reversas', dropna=False)
        .agg(
            produtos=('product_name', 'nunique'),
            reversas_ultimos_3m=('reversas_ultimos_3m', 'sum'),
            reversas_3m_anteriores=('reversas_3m_anteriores', 'sum'),
            itens_retornados=('qt_items_returned', 'sum'),
        )
        .reset_index()
        .sort_values('itens_retornados', ascending=False)
        .reset_index(drop=True)
    )
    display(df_trend)
else:
    print('Coluna tendencia_reversas nao encontrada no dataframe')


,tendencia_reversas,produtos,reversas_ultimos_3m,reversas_3m_anteriores,itens_retornados
0,Estável,40,19261,19469,"82,030.0000"
1,Caiu nos últimos 3 meses,30,8733,14481,"56,587.0000"
2,Aumentou nos últimos 3 meses,33,13043,6051,"32,228.0000"
3,Apareceu nos últimos 3 meses,7,281,0,289.0000
4,Sem volume para tendência,4,4,1,54.0000
5,NaN,1,0,0,0.0000


### 8. Concentração de Cor e Tamanho nos Top Produtos

In [16]:
df_prep_top = df_prep[
    df_prep['sinal_priorizacao'].isin(['Priorizar melhoria', 'Alerta em produto relevante'])
].copy()

cols_conc = [
    'product_name', 'sinal_priorizacao',
    'principal_cor_afetada', 'principal_cor_pct',
    'principal_tamanho_afetado', 'principal_tamanho_pct',
    'td_rate', 'qt_items_returned',
]
available_conc = [c for c in cols_conc if c in df_prep_top.columns]

print(f'Produtos em alerta com concentracao de cor/tamanho:')
has_cor = df_prep_top.get('principal_cor_pct', pd.Series(dtype=float)) >= 0.50
has_tam = df_prep_top.get('principal_tamanho_pct', pd.Series(dtype=float)) >= 0.50
df_conc = df_prep_top[has_cor | has_tam]
print(f'  {len(df_conc)} produto(s) com concentracao >= 50% em cor ou tamanho')
display(df_conc[available_conc].reset_index(drop=True))

Produtos em alerta com concentracao de cor/tamanho:
  11 produto(s) com concentracao >= 50% em cor ou tamanho


,product_name,sinal_priorizacao,principal_cor_afetada,principal_cor_pct,principal_tamanho_afetado,principal_tamanho_pct,td_rate,qt_items_returned
0,Calça FutureForm Feminino,Priorizar melhoria,Preto,0.5419,M,0.3508,0.1650,"4,293.0000"
1,Tube Dress Feminino,Priorizar melhoria,Preto,1.0000,M,0.3908,0.1717,"3,514.0000"
2,Skin Cropped Long Sleeve Feminino,Priorizar melhoria,Preto,0.7183,M,0.3953,0.1588,"2,961.0000"
3,Saia Mini Kyoto Feminino,Priorizar melhoria,Preto,0.5363,M,0.3780,0.2045,"2,315.0000"
4,Camisa FutureForm Feminino,Priorizar melhoria,Off White,0.5423,M,0.3731,0.1389,"3,023.0000"
5,Tech T-shirt Heavy Slim Masculino,Priorizar melhoria,Preto,0.5103,M,0.3380,0.1060,"2,109.0000"
6,Saia Midi Kyoto Feminino,Priorizar melhoria,Preto,0.5079,M,0.3362,0.1331,"1,396.0000"
7,Maxi Saia NYIN Feminino,Alerta em produto relevante,Preto,0.5256,P,0.2913,0.0917,"3,376.0000"
8,Tech T-shirt Heavy Masculino,Alerta em produto relevante,Preto,0.5021,M,0.2772,0.0703,"2,443.0000"
9,Saia Envelope Breeze Feminino,Alerta em produto relevante,Preto,0.6736,P/M,0.7562,0.1125,"1,701.0000"


### 9. Sinal Consolidado (Regra QA) — Visão Produto

Cruza `commercial_score` (proxy de Tração Comercial, Opção B) com `td_score` para aplicar a regra de dois estados da spec QA. Exibe o Top 3 de motivos em coluna HTML `top_3_motivos_td` — pronto para a aba **relatorio_pf** do workbook compartilhado com o time PF.

In [17]:
# ── Busca pilares do scorecard em sop_bronze.eval_produto_portfolio ──────────
# Fonte: score_vendas_geral, score_viabilidade_financeira, score_satisf_cliente
# Escala original 0-1 → ×100. Grão: product_name.
# Nota: eval_produto_portfolio NÃO tem coluna cluster — cluster vem de portfolio_skp_clustering.

SQL_SCORECARD = """
  SELECT
    product_name,
    ROUND(ANY_VALUE(score_vendas_geral) * 100, 1)           AS score_tracao_comercial,
    ROUND(ANY_VALUE(score_viabilidade_financeira) * 100, 1)  AS score_unit_economics,
    ROUND(ANY_VALUE(score_satisf_cliente) * 100, 1)          AS score_satisfacao_marca
  FROM `insider-data-lake.sop_bronze.eval_produto_portfolio`
  GROUP BY product_name
"""

print('Executando query de scorecard (eval_produto_portfolio)...')
df_scores_raw = client.query(SQL_SCORECARD).to_dataframe()
print(f'  {len(df_scores_raw):,} produto(s) com scorecard')

# Adiciona td_produto_pct e td_categoria_pct para build_scorecard_product_view
_td_ref = (
    df_prep[['product_name', 'td_rate', 'td_rate_categoria']]
    .drop_duplicates('product_name')
    .rename(columns={'td_rate': 'td_produto_pct', 'td_rate_categoria': 'td_categoria_pct'})
)

# Cluster vem de portfolio_skp_clustering (já em df_prep via portfolio_cluster)
_cluster_ref = (
    df_prep[['product_name', 'portfolio_cluster']]
    .drop_duplicates('product_name')
    .rename(columns={'portfolio_cluster': 'cluster'})
)

scorecard_df_real = (
    df_scores_raw
    .merge(_td_ref, on='product_name', how='left')
    .merge(_cluster_ref, on='product_name', how='left')
)

_coverage = scorecard_df_real['score_tracao_comercial'].notna().sum()
print(f'  Scores populados: {_coverage}/{len(scorecard_df_real)} produto(s)')
print(f'  Produtos sem scorecard (ficarão com —): '
      f'{len(scorecard_df_real) - _coverage}')

df_scorecard_view = build_scorecard_product_view(
    df_prep,
    scorecard_df_real,
    only_buckets=['Priorizar melhoria', 'Alerta em produto relevante', 'Monitorar'],
)

print(f'\nVisão produto (relatorio_pf): {len(df_scorecard_view)} produto(s)')
display(df_scorecard_view.head(30))


Executando query de scorecard (eval_produto_portfolio)...


/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  126 produto(s) com scorecard
  Scores populados: 126/126 produto(s)
  Produtos sem scorecard (ficarão com —): 0

Visão produto (relatorio_pf): 43 produto(s)


,product_name,category,gender,cluster,sinal_priorizacao,td_score,score_tracao_comercial,score_unit_economics,score_satisfacao_marca,td_categoria_pct,td_produto_pct,td_rate,td_rate_categoria,tendencia_reversas,top_3_motivos_td,tweet_analitico,priority_score,qt_items_returned,receita_liquida
0,Calça FutureForm Feminino,Calça,female,Hero,Priorizar melhoria,0.9200,78.6000,19.5000,11.1000,0.1230,0.1650,0.1650,0.1230,Estável,1. caimento_ruim (14%)<br>2. modelagem_ruim (1...,"Ofensores: caimento_ruim (14%), modelagem_ruim...",0.9054,"4,293.0000","8,996,127.2200"
1,Tube Dress Feminino,Vestido,female,Core,Priorizar melhoria,0.8861,79.5000,11.6000,21.4000,0.1634,0.1717,0.1717,0.1634,Caiu nos últimos 3 meses,1. tamanho_pequeno (16%)<br>2. caimento_ruim (...,"Ofensores: tamanho_pequeno (16%), caimento_rui...",0.8628,"3,514.0000","5,699,452.9000"
2,Skin Cropped Long Sleeve Feminino,Cropped,female,Long tail,Priorizar melhoria,0.9113,53.4000,69.5000,9.5000,0.1043,0.1588,0.1588,0.1043,Aumentou nos últimos 3 meses,1. tamanho_pequeno (14%)<br>2. modelagem_ruim ...,"Ofensores: tamanho_pequeno (14%), modelagem_ru...",0.8264,"2,961.0000","2,178,086.7000"
3,Saia Mini Kyoto Feminino,Saia,female,Core,Priorizar melhoria,0.9322,56.4000,31.1000,10.2000,0.1190,0.2045,0.2045,0.1190,Aumentou nos últimos 3 meses,1. caimento_ruim (13%)<br>2. modelagem_ruim (8...,"Ofensores: caimento_ruim (13%), modelagem_ruim...",0.8243,"2,315.0000","2,620,113.0900"
4,Calça Wide Leg Feminino,Calça,female,Core,Priorizar melhoria,0.8130,58.7000,49.6000,17.6000,0.1230,0.1303,0.1303,0.1230,Aumentou nos últimos 3 meses,1. caimento_ruim (8%)<br>2. modelagem_ruim (6%...,"Ofensores: caimento_ruim (8%), modelagem_ruim ...",0.8134,"2,599.0000","5,486,871.5500"
5,Core T-shirt Masculino,T-shirt,male,NaN,Priorizar melhoria,0.7052,NaN,NaN,NaN,NaN,NaN,0.0779,0.0550,Estável,1. tamanho_pequeno (6%)<br>2. tecido_grosso (3...,"Ofensores: tamanho_pequeno (6%), tecido_grosso...",0.8127,"8,429.0000","13,981,701.9200"
6,Camisa FutureForm Feminino,Camisa Social,female,Core,Priorizar melhoria,0.7896,72.2000,23.4000,44.4000,0.1421,0.1389,0.1389,0.1421,Aumentou nos últimos 3 meses,1. tamanho_grande (11%)<br>2. cor_diferente_si...,"Ofensores: tamanho_grande (11%), cor_diferente...",0.8118,"3,023.0000","7,947,348.6700"
7,Shorts Kyoto Feminino,Shorts,female,Core,Priorizar melhoria,0.8522,67.3000,57.2000,27.1000,0.1841,0.1841,0.1841,0.1841,Estável,1. caimento_ruim (11%)<br>2. modelagem_ruim (8...,"Ofensores: caimento_ruim (11%), modelagem_ruim...",0.8077,"2,906.0000","3,425,742.1200"
8,Camisa FutureForm Masculino,Camisa Social,male,Core,Priorizar melhoria,0.8383,67.1000,13.9000,45.8000,0.1421,0.1522,0.1522,0.1421,Aumentou nos últimos 3 meses,1. tamanho_grande (13%)<br>2. caimento_ruim (4...,"Ofensores: tamanho_grande (13%), caimento_ruim...",0.8077,"2,219.0000","5,173,195.0200"
9,The Perfect Top Feminino,Regata,female,Long tail,Alerta em produto relevante,0.6357,95.0000,20.6000,60.7000,0.0766,0.0749,0.0749,0.0766,Estável,1. tamanho_pequeno (6%)<br>2. caimento_ruim (6...,"Ofensores: tamanho_pequeno (6%), caimento_ruim...",0.7779,"23,507.0000","32,806,350.4900"


### 10. Tweet Analista — Relatório Executivo

Gera um resumo analítico por produto combinando: Top 3 motivos (com % de alavanca), concentração de cor/tamanho (se ≥ 50%), taxa T&D vs categoria e tendência recente. Fonte: heurística baseada em regras — colunas pré-computadas pelo SQL, sem dependência de LLM.

In [18]:
# Gera contexto LLM + tweet heurístico por produto
llm_rows = build_llm_context_rows(
    df_prep,
    only_buckets=('Priorizar melhoria', 'Alerta em produto relevante', 'Monitorar'),
    max_rows=50,
)

for row in llm_rows:
    row['tweet_analista'] = build_tweet_heuristic(row)

print(f'{len(llm_rows)} produto(s) com tweet analista gerado')


43 produto(s) com tweet analista gerado


In [19]:
# Relatório executivo — formato spec QA
BUCKETS_RELATORIO = ['Priorizar melhoria', 'Alerta em produto relevante', 'Monitorar']
tweet_map = {r['product_name']: r.get('tweet_analista', '—') for r in llm_rows}

df_report = (
    df_prep[df_prep['sinal_priorizacao'].isin(BUCKETS_RELATORIO)]
    .sort_values('priority_score', ascending=False)
)

for _, row in df_report.iterrows():
    sinal  = row.get('sinal_kill_keep') or row.get('sinal_priorizacao') or '—'
    t1     = row.get('top_1_problema') or '—'
    t2     = row.get('top_2_problema') or '—'
    t3     = row.get('top_3_problema') or '—'
    tweet  = tweet_map.get(row['product_name'], '—')

    print(f"\n{'─' * 90}")
    print(f"  {row['product_name']}  —  Sinal: {sinal}")
    print(f"  • Top 3 problemas: {t1}, {t2} e {t3}")
    print(f"  • Tweet: {tweet}")

print(f"\n{'─' * 90}")
print(f"Total: {len(df_report)} produto(s) no relatório")



──────────────────────────────────────────────────────────────────────────────────────────
  Calça FutureForm Feminino  —  Sinal: Priorizar melhoria
  • Top 3 problemas: caimento_ruim, modelagem_ruim e tamanho_pequeno
  • Tweet: Ofensores: caimento_ruim (14%), modelagem_ruim (12%), tamanho_pequeno (5%). Cor crítica: Preto (54% das reversas). T&D: 16.5% produto vs 12.3% categoria (4.2%pp acima). Top 3 concentra 32% das reversas tagueadas. Ação sugerida: Modelagem.

──────────────────────────────────────────────────────────────────────────────────────────
  Tube Dress Feminino  —  Sinal: Priorizar melhoria
  • Top 3 problemas: tamanho_pequeno, caimento_ruim e modelagem_ruim
  • Tweet: Ofensores: tamanho_pequeno (16%), caimento_ruim (12%), modelagem_ruim (7%). Cor crítica: Preto (100% das reversas). T&D: 17.2% produto vs 16.3% categoria (0.8%pp acima). Top 3 concentra 34% das reversas tagueadas. Tendência recente: Caiu nos últimos 3 meses. Ação sugerida: Grade.

─────────────────────────

## Workbook — Design e Entrega

### Requisitos (`sheets_writer` skill — obrigatórios para todos os `.xlsx`)

| Requisito | Status | Detalhe |
|---|---|---|
| **Fonte profissional** | ✅ Arial 10 | Aplicada via `_apply_workbook_formatting()` em todos os headers |
| **Zero erros de fórmula** | ✅ N/A | Arquivo data-driven — sem fórmulas Excel; `#REF!`, `#DIV/0!` não se aplicam |
| **Library** | ✅ pandas + openpyxl | `pandas` para dados BigQuery · `openpyxl` para formatação e cores |
| **recalc.py** | ✅ Não requerido | Dados pré-computados no BigQuery — sem fórmulas a recalcular |

**Fonte de dados:** `insider-data-lake` (BigQuery) · queries em `pipeline_reversas_priorizacao_produtos.sql` · sem hardcodes de valores calculados.

---

### Abas do arquivo `td_priorizacao_YYYYMMDD.xlsx`

| Aba | Propósito | Audiência |
|---|---|---|
| `farol_executivo` | Tabela completa — todos os produtos × 44 colunas com scores, métricas, tags e sinais | Analytics / referência técnica |
| `resumo_farol` | Agregação por bucket: contagem, receita líquida e share de T&D | Executivo / reuniões de portfólio |
| `benchmark_categoria` | Taxa de T&D por categoria — mediana, total e produtos em prioridade | Analytics / contexto de categoria |
| `resumo_tags` | Top tags globais por prevalência e concentração média por produto | PF / patterns sistêmicos |
| `priorizar_melhoria` | Top produtos com ação imediata, ordenados por `priority_score` | Time PF — foco de sprint |
| **`tweets_analiticos`** | **Entregável principal** — `mensagem_consolidada` por produto (Priorizar + Alerta), ordenados bucket → score | Time PF — consumo direto |
| `relatorio_pf` | Scorecard de portfólio (Tração Comercial, Unit Economics, Satisfação/Marca) + Top 3 motivos HTML | Time PF — análise contextual |

---

### Codificação de Cores (convenção de workbook de dados — background de linha)

> ⚠️ Esta convenção usa **cor de fundo por linha** (não texto azul/preto/verde da convenção de modelo financeiro). Padrão aprovado no design de governança.

| Cor de fundo | Bucket | Critério |
|---|---|---|
| 🔴 `#FFDEDE` — Vermelho claro | **Priorizar melhoria** | `td_score ≥ 0.70` e `commercial_score ≥ 0.60` |
| 🟡 `#FFFACD` — Amarelo claro | **Alerta em produto relevante** | `commercial_score ≥ 0.70` e `0.50 ≤ td_score < 0.70` |
| 🟠 `#FFE5CC` — Laranja claro | **Monitorar** | `td_score ≥ 0.70` e `commercial_score < 0.60` |
| ⚪ `#F0F0F0` — Cinza claro | **Não priorizar agora** | `td_score < 0.50` ou `commercial_score < 0.40` |

**Headers:** fundo `#1F497D` (azul marinho) · texto branco bold · Arial 10 · freeze linha 1.  
**Abas coloridas:** vermelho escuro (`tweets_analiticos`, `priorizar_melhoria`) · azul (`farol_executivo`) · verde (`resumo_farol`) · laranja (`relatorio_pf`).

---

### Formato da `mensagem_consolidada` (aba `tweets_analiticos`)

Bloco de 3 linhas pronto para consumo direto pelo time PF:

```
{produto} - Sinal: {bucket} ({descrição do sinal})
Top 3 problemas: {tag1}, {tag2} e {tag3}
Tweet: {resumo analítico com tipo de ação sugerida}
```

**Exemplos de descrição por sinal:**
- `Priorizar melhoria` → *alta T&D e alta tração comercial*
- `Alerta em produto relevante` → *produto relevante com T&D em escalada*
- `Monitorar` → *alta T&D, baixa tração comercial*

**Tipo de ação** (coluna `tipo_de_acao`, derivado da Top 1 tag):  
`Modelagem` · `Grade` · `Tecido` · `PDP/Comunicação` · `Investigação adicional` · `Investigar tags negativas` · `Logístico — fora do escopo PF`

---

### Checklist pré-entrega

- [ ] Fonte Arial 10 aplicada em todos os cabeçalhos
- [ ] Zero células com erros (`#REF!`, `#DIV/0!`, `#VALUE!`, `#N/A`, `#NAME?`)
- [ ] `mensagem_consolidada` gerada para todos os produtos (Priorizar + Alerta)
- [ ] Cores de linha corretas por `sinal_priorizacao` em todas as abas
- [ ] Aba `tweets_analiticos` visível e com `mensagem_consolidada` como coluna principal
- [ ] `relatorio_pf` contém os 3 pilares do scorecard (Tração, Unit Economics, Satisfação)

## Export

In [20]:
# Excel workbook com abas: farol_executivo, resumo_farol, benchmark_categoria,
#   resumo_tags, priorizar_melhoria, tweets_analiticos, relatorio_pf,
#   (amostra_analitica se Camada 1 carregada)
xlsx_path = str(OUTPUT_DIR / f'td_priorizacao_{DATE_TAG}.xlsx')

df_tweets = build_tweets_tab(df_exec)
print(f'Tweets tab: {len(df_tweets)} produto(s) | '
      f'{(df_tweets["sinal_priorizacao"] == "Priorizar melhoria").sum()} Priorizar + '
      f'{(df_tweets["sinal_priorizacao"] == "Alerta em produto relevante").sum()} Alerta')

export_priority_workbook(
    df_exec, xlsx_path,
    analytical_df=df_analitica,
    scorecard_view_df=df_scorecard_view,
    tweets_df=df_tweets,
)
print(f'Workbook: {xlsx_path}')

# CSV do farol executivo
csv_path = str(OUTPUT_DIR / f'td_priorizacao_{DATE_TAG}.csv')
prepare_executive_df(df_exec).to_csv(csv_path, index=False)
print(f'CSV: {csv_path}')

# Preview LLM context — primeiros 5 campos do produto #1
_llm_preview = build_llm_context_rows(df_exec, max_rows=5)
print(f'\nLLM context preview: {len(_llm_preview)} produto(s)')
if _llm_preview:
    print('Exemplo — produto #1:')
    for k, v in list(_llm_preview[0].items())[:8]:
        if v is not None:
            print(f'  {k}: {v}')


Tweets tab: 25 produto(s) | 12 Priorizar + 13 Alerta


Workbook: /Users/insider/LA_Coding_Projects/outputs/relatorio_td/td_priorizacao_20260612.xlsx
CSV: /Users/insider/LA_Coding_Projects/outputs/relatorio_td/td_priorizacao_20260612.csv

LLM context preview: 5 produto(s)
Exemplo — produto #1:
  product_name: Calça FutureForm Feminino
  category: Calça
  gender: female
  portfolio_cluster: Hero
  sinal_priorizacao: Priorizar melhoria
  td_rate: 0.16500115304788993
  td_rate_categoria: 0.12295430155607226
  delta_vs_categoria: 0.042046851491817666


In [21]:
df_tags = client.query("""
    SELECT DISTINCT tag
    FROM `insider-data-lake.sop_silver.return_reason_tags`,
    UNNEST(tags) AS tag
    WHERE tag IS NOT NULL
    ORDER BY tag
""").to_dataframe()

print(f"{len(df_tags)} tags distintas encontradas:\n")
for t in df_tags['tag']:
    print(f"  {t}")


38 tags distintas encontradas:

  atendimento_ineficiente
  caimento_bom
  caimento_ruim
  comprimento_curto
  comprimento_longo
  conforto_negativo
  conforto_positivo
  cor_diferente_site
  defeito_aviamento
  defeito_costura
  defeito_fio_puxado
  defeito_furo_rasgo
  defeito_gola
  defeito_mancha
  encolhimento
  feedback_positivo_geral
  logistica_adiantamento
  logistica_atraso
  logistica_embalagem
  logistica_item_errado
  logistica_item_faltando
  modelagem_boa
  modelagem_ruim
  pilling_bolinhas
  provador_virtual_impreciso
  sustentacao_ruim
  tamanho_grande
  tamanho_ideal
  tamanho_pepequeno
  tamanho_pequeno
  tecido_amassa
  tecido_fino
  tecido_grosso
  tecido_marca_corpo
  tecido_qualidade_boa
  tecido_qualidade_ruim
  tecido_quente
  tecido_transparente


/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [22]:
# ── Upload Excel → Google Sheets via gcloud user credentials ─────────────────
# Não requer nenhuma permissão no GCP Console.
#
# PRÉ-REQUISITO (já feito — rode no terminal se precisar de novo):
#   gcloud auth login --enable-gdrive-access
#
# Usa o token do gcloud (que já inclui escopo drive) para autenticar no gspread.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess
import gspread
import google.oauth2.credentials
from gspread_dataframe import set_with_dataframe

# Obtém access token do gcloud (inclui Drive scope)
result = subprocess.run(
    ['gcloud', 'auth', 'print-access-token'],
    capture_output=True, text=True, check=True,
)
access_token = result.stdout.strip()
creds = google.oauth2.credentials.Credentials(token=access_token)
gc = gspread.Client(auth=creds)
print('Autenticado via gcloud user credentials (Drive scope)')

# Lê abas do Excel gerado
xl = pd.ExcelFile(xlsx_path)
sheet_names = xl.sheet_names
print(f'Abas a enviar: {sheet_names}')

# Cria novo Google Sheets
ss_title = f'TD Priorização Melhorias {DATE_TAG}'
spreadsheet = gc.create(ss_title)
print(f'\nSpreadsheet criado: {spreadsheet.url}')

# Upload aba por aba
for i, sheet_name in enumerate(sheet_names):
    df_sheet = xl.parse(sheet_name).fillna('')
    if i == 0:
        ws = spreadsheet.sheet1
        ws.update_title(sheet_name)
    else:
        ws = spreadsheet.add_worksheet(
            title=sheet_name,
            rows=max(len(df_sheet) + 10, 100),
            cols=max(len(df_sheet.columns) + 5, 30),
        )
    set_with_dataframe(ws, df_sheet)
    print(f'  ✓ {sheet_name}: {len(df_sheet):,} linhas × {len(df_sheet.columns)} colunas')

print(f'\n✅ Upload completo → {spreadsheet.url}')


Autenticado via gcloud user credentials (Drive scope)
Abas a enviar: ['farol_executivo', 'resumo_farol', 'benchmark_categoria', 'resumo_tags', 'priorizar_melhoria', 'tweets_analiticos', 'relatorio_pf']



Spreadsheet criado: https://docs.google.com/spreadsheets/d/1DFhjwF1ZtyVotOSw3dQ2xJ_MZZA7vknZljzplR9LXws


  ✓ farol_executivo: 115 linhas × 44 colunas


  ✓ resumo_farol: 5 linhas × 9 colunas


  ✓ benchmark_categoria: 24 linhas × 9 colunas


  ✓ resumo_tags: 18 linhas × 4 colunas


  ✓ priorizar_melhoria: 12 linhas × 44 colunas


  ✓ tweets_analiticos: 25 linhas × 11 colunas


  ✓ relatorio_pf: 43 linhas × 19 colunas

✅ Upload completo → https://docs.google.com/spreadsheets/d/1DFhjwF1ZtyVotOSw3dQ2xJ_MZZA7vknZljzplR9LXws
